In [ ]:
import h5py
import pandas as pd
import numpy as np
import os
from pathlib import Path
from matplotlib import pyplot as plt
import scipy.stats
from scipy.stats import wilcoxon
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, Polygon
from matplotlib.collections import PolyCollection
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from functools import partial
import warnings
import vbn_utils
import decoding_utils as du
from analysis_utils import exponential_convolve
import ccf_utils
from vbn_utils import cumulative_hist, formatFigure, mean_sem_plot, make_iterable, get_unit_ids, bootstrap_ci
%matplotlib inline

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv" #"/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.5.0/project_metadata/ecephys_sessions.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'] = units['cortical_layer'].replace('3-Feb','2/3') # necessary since 2/3 sometimes gets incorrectly reformatted as a date

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

active_tensor = h5py.File(active_tensor_file)

sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
new_clusters = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/cluster_labels_122024.csv")
units = units.merge(new_clusters[['unit_id', 'cluster_labels_new_weak_cluster_12', 'used_in_clustering']], on='unit_id', how='left')
units.rename(columns={'cluster_labels_new_weak_cluster_12': 'cluster_labels_new', 'used_in_clustering': 'used_in_new_clustering'}, inplace=True)

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/ccf_structure_tree_2017.csv")

In [ ]:
stim_table['is_shared'] = stim_table['image_name'].isin(['im083_r', 'im111_r'])

## Time from last lick

### Cluster-wise responses as function of time from last lick (running matched)

In [ ]:
running_df = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/all_running_dfs.csv")

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 15

In [ ]:
import warnings
warnings.filterwarnings("ignore")

plt.rcParams['font.size'] = 15

fig_agg, ax_agg = plt.subplots()

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'coral',  # Lighter red
    8: 'gold'   # Lightest red
}

cluster_names = {
	1: 'On Transient',
	2: 'On Sustained',
	3: 'Off Transient',
	4: 'Off Sustained',
	5: 'Image suppressed',
	6: 'Lick anticipation',
	7: 'Licking',
	8: 'Running'
}

time = np.arange(-750,750)

all_cluster_means = {}
for cluster in range(6,8):
	fig, ax = plt.subplots()
	fig.suptitle(cluster_names[cluster])
	unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
			du.getUnitsInRegion(units, 'SCMRN')
			
	unit_ids = units.loc[unit_filter]['unit_id'].values
	if len(unit_ids)==0:
		continue
	session_list = units.loc[unit_filter]['ecephys_session_id'].unique()

	max_flashes = 7

	stim_filter_base = ['engaged', '~is_change', '~omitted', '~previous_omitted', '~grace_period_after_hit']# 'flashes_since_change>5']
	cond_filters = [
			['engaged', 'is_change', 'hit'],
			['lickbout_for_flash_during_response_window'] + stim_filter_base,
			['engaged', 'is_change', 'miss']
			] + [[f'flashes_since_last_lick=={flashes_since_lick}', '~lickbout_for_flash_during_response_window'] + 
					stim_filter_base for flashes_since_lick in range(2, max_flashes+1)]

	unit_data, stim_indices, unitIds, returned_session_ids = vbn_utils.unit_averaged_psth_col_matched(active_tensor_file, stim_table, session_list, unit_ids, cond_filters, 'baseline_running', 
							baseline_length=750, resp_window_length=750, num_iterations=1)
	
	passive_unit_data, passive_stim_indices, passive_unitIds, passive_returned_session_ids = vbn_utils.unit_averaged_psth_col_matched(passive_tensor_file, stim_table, session_list, unit_ids, cond_filters, 'baseline_running', 
							baseline_length=750, resp_window_length=750, num_iterations=1)
	
	unit_data = [u for u in unit_data if u.size>0]
	unit_data = np.concatenate(unit_data, axis=1)
	passive_unit_data = [u for u in passive_unit_data if u.size>0]
	passive_unit_data = np.concatenate(passive_unit_data, axis=1)
	cluster_means = []
	for icond, cond in enumerate(unit_data):
		if icond==0:
			label = 'change lick'
			alpha = 1
			color = 'g'
			ls = 'solid'
		
		elif icond==1: 
			label = 'non change lick'
			color = 'g'
			alpha = 1
			ls = 'dotted'
		
		elif icond== 2:
			label = 'miss'
			alpha = 1
			color = 'goldenrod'
			ls = 'solid'

		elif icond>2:
			label = f'{icond-1} flashes since lick'
			alpha = (icond-1)/max_flashes
			color = 'k'
			ls = 'solid'
		
		
		
		cond = np.array([exponential_convolve(c, 7, symmetrical=True) for c in cond])
		ax.plot(time, np.nanmean(cond, axis=0)*1000, color=color, alpha=alpha, ls=ls,)# label=icond)
		cluster_means.append(np.nanmean(cond[:, 550:750]*1000, axis=1))
    
	cluster_means = np.array(cluster_means)
	cluster_means = np.roll(cluster_means, shift=-3, axis=0)
	all_cluster_means[cluster] = cluster_means
	cluster_means = cluster_means - cluster_means[0][None,:]
	cluster_mean = np.nanmean(cluster_means, axis=1)
	cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
	
	ax_agg.plot(np.arange(2,max_flashes+1), cluster_mean[:-3], 'o', color=cluster_colors[cluster], label=cluster_names[cluster])
	ax_agg.errorbar(np.arange(2,max_flashes+1), cluster_mean[:-3], yerr=cluster_sem[:-3], color=cluster_colors[cluster])
	ax_agg.plot(np.arange(max_flashes+1,max_flashes+4), cluster_mean[-3:], 'o', color=cluster_colors[cluster],)# label=cluster)
	ax_agg.errorbar(np.arange(max_flashes+1,max_flashes+4), cluster_mean[-3:], yerr=cluster_sem[-3:], color=cluster_colors[cluster])
	
	ax.set_xlim(-300, 500)
	ax.set_xlabel('Time from image onset (ms)')
	ax.set_ylabel('Firing rate (Hz)')

	passive_hit = np.array([exponential_convolve(c, 7, symmetrical=True) for c in passive_unit_data[0]])
	ax.plot(time, np.nanmean(passive_hit, axis=0)*1000, color='g', alpha=0.5, ls='-.', label='passive hit')
	ax.legend(['hit', 'false alarm', 'miss'] + list(np.arange(2,max_flashes+1)) + ['passive hit',], frameon=False, loc='upper left', bbox_to_anchor=(0, 1.05))


	vbn_utils.formatFigure(fig, ax)

ax_agg.set_xticks(np.arange(2,max_flashes+4))
ax_agg.set_xticklabels(list(np.arange(2,max_flashes+1)) + ['hit', 'false alarm', 'miss'])
for il, label in enumerate(ax_agg.get_xticklabels()):
	if il > max_flashes-2:
		label.set_rotation(30)

ax_agg.legend(frameon=False)
ax_agg.set_xlabel('Image presentations since last lick')
ax_agg.set_ylabel('Baseline firing rate (Hz)')
vbn_utils.formatFigure(fig_agg, ax_agg)

## Analysis of behavior since last lick

In [ ]:
stims = stim_table[stim_table['no_abnorm']]

In [ ]:
fig, ax = plt.subplots()
table = stims[stims['engaged']]
ax.plot(table.pivot_table(index='flashes_since_last_lick', values='is_change', aggfunc='mean'), 'ko-',)
ax.set_xlim(0, 15.5)
ax.set_xlabel('Image presentations since last lick')
ax.set_ylabel('change probability')
ax.set_ylim(-0.01,0.35)

ax2 = ax.twinx()
table = stims[(~stims['is_change'])&(~stims['omitted'])&(stims['engaged'])&(~stims['previous_omitted'])]
ax2.plot(table.pivot_table(index='flashes_since_last_lick', 
            values='lickbout_for_flash_during_response_window', aggfunc='mean'), 'ro-',)
ax2.set_xlim(0, 15.5)
ax2.set_ylabel('false alarm rate', color='r')
ax2.set_ylim(-0.01,0.35)
ax2.tick_params(axis='y', colors='r')

[a.spines['top'].set_visible(False) for a in [ax, ax2]]

In [ ]:
stims['rt_zscore'] = stims.groupby('session_id')['reaction_time'].transform(lambda x: (x-np.nanmean(x))/np.nanstd(x))

fig, ax = plt.subplots()
hit_table = stims[stims['is_change']&stims['engaged']&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = hit_table.pivot_table(index='flashes_since_last_lick', values='rt_zscore', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='k', ax=ax, label='hits')


fa_table = stims[(stims['engaged'])&(~stims['is_change'])&(~stims['omitted'])&(~stims['previous_omitted'])&(~stims['grace_period_after_hit']) &\
    (stims['flashes_since_change']>1)&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = fa_table.pivot_table(index='flashes_since_last_lick', values='rt_zscore', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='r', ax=ax, label='false alarms')
ax.set_xlim(0,15.5)
ax.set_ylim(-0.55,0.5)
ax.set_xlabel('Image presentations since last lick')
ax.set_ylabel('z-scored response time')

plt.legend()
vbn_utils.formatFigure(fig, ax)

fig, ax = plt.subplots()
hit_table = stims[stims['is_change']&stims['engaged']&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = hit_table.pivot_table(index='flashes_since_last_lick', values='reaction_time', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='k', ax=ax, label='hits')


fa_table = stims[(stims['engaged'])&(~stims['is_change'])&(~stims['omitted'])&(~stims['previous_omitted'])&(~stims['grace_period_after_hit']) &\
    (stims['flashes_since_change']>1)&(stims['flashes_since_last_lick']>1)&(stims['flashes_since_last_lick']<16)]
vals = fa_table.pivot_table(index='flashes_since_last_lick', values='reaction_time', aggfunc=('mean', 'sem'))
vals.plot(y='mean', yerr='sem', kind='line', marker='o', color='r', ax=ax, label='false alarms')
ax.set_xlim(0,15.5)
ax.set_xlabel('Image presentations since last lick')
ax.set_ylabel('Response time (s)')
ax.set_ylim(0.35, 0.5)
plt.legend()
vbn_utils.formatFigure(fig, ax)


## Responses by reaction time quintile

### Running matched

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig_rt, ax_rt = plt.subplots()
for q in range(5):

    rts = stim_table[stim_table['rt_quintiles']==q]['reaction_time'].values
    ax_rt.boxplot(rts, notch=True, positions=[q,], showfliers=False)
ax_rt.set_xticks(np.arange(5))
ax_rt.set_xticklabels(np.arange(1,6))
ax_rt.set_ylabel('Response time (s)')
ax_rt.set_xlabel('Response time quintile')
vbn_utils.formatFigure(fig_rt, ax_rt)

fig_agg, ax_agg = plt.subplots()

cluster_colors = {
    1: 'navy',  # Blue
    2: 'mediumblue',  # Lighter blue
    3: 'royalblue',  # Even lighter blue
    4: 'cornflowerblue',  # Lighter blue
    5: 'slateblue',  # Lightest blue
    6: 'red',  # Red
    7: 'coral',  # Lighter red
    8: 'gold'   # Lightest red
}

cluster_names = {
	1: 'On Transient',
	2: 'On Sustained',
	3: 'Off Transient',
	4: 'Off Sustained',
	5: 'Image suppressed',
	6: 'Lick anticipation',
	7: 'Licking',
	8: 'Running'
}

time = np.arange(-750, 750)
for cluster in range(6,8):
	fig, ax = plt.subplots()
	fig.suptitle(cluster_names[cluster])
	# fig.set_size_inches(12,6)
	unit_filter = du.get_units_in_cluster(units, cluster, clustering='new') & du.apply_unit_quality_filter(units) & \
			du.getUnitsInRegion(units, 'SCMRN')
			
	unit_ids = units.loc[unit_filter]['unit_id'].values
	if len(unit_ids)==0:
		continue
	session_list = units.loc[unit_filter]['ecephys_session_id'].unique()

	max_flashes = 7

	stim_filter_base = ['engaged', 'is_change', '~omitted', '~previous_omitted',]# 'flashes_since_change>5']
	cond_filters = [
			['engaged', 'is_change', 'hit'],
			['engaged', '~is_change', '~omitted', '~previous_omitted',
                '~lickbout_for_flash_during_response_window', 'flashes_since_last_lick>1', 'flashes_since_change>5'],
			] + [[f'rt_quintiles=={rt_quintile}', 'lickbout_for_flash_during_response_window'] + 
					stim_filter_base for rt_quintile in range(5)]

	unit_data, stim_indices, unitIds, returned_session_ids = vbn_utils.unit_averaged_psth_col_matched(active_tensor_file, 
							stim_table, session_list, unit_ids, cond_filters, 'baseline_running', 
							baseline_length=750, resp_window_length=750, num_iterations=5)
	
	unit_data = [u for u in unit_data if u.size>0]
	unit_data = np.concatenate(unit_data, axis=1)
	cluster_means = []
	for icond, cond in enumerate(unit_data):
		if icond==0:
			label = 'hit'
			alpha = 0
			color = 'g'
			ls = 'solid'
		
		elif icond==1: 
			label = 'correct reject'
			color = 'g'
			alpha = 1
			ls = 'dotted'
		
		elif icond>=2:
			label = f'{icond-2}'
			alpha = 1.2-(icond-1)/5
			color = 'k'
			ls = 'solid'
		
		cond = np.array([exponential_convolve(c, 7, symmetrical=True) for c in cond])
		if icond>0:
			ax.plot(time, np.nanmean(cond, axis=0)*1000, color=color, alpha=alpha, ls=ls, label=label)
		cluster_means.append(np.nanmean(cond[:, 550:750]*1000, axis=1))
    
	ax.set_xlabel('Time from flash (ms)')

	cluster_means = np.array(cluster_means)
	cluster_means = np.roll(cluster_means, shift=-2, axis=0)
	cluster_means = cluster_means - cluster_means[0][None,:]

	cluster_means_norm = cluster_means/np.nanmax(np.abs(cluster_means), axis=0)[None, :]

	cluster_mean = np.nanmean(cluster_means, axis=1)
	cluster_sem = np.nanstd(cluster_means, axis=1)/cluster_means.shape[1]**0.5
	
	ax_agg.plot(np.arange(5), cluster_mean[:-2], 'o', color=cluster_colors[cluster], label=cluster_names[cluster])
	ax_agg.errorbar(np.arange(5), cluster_mean[:-2], yerr=cluster_sem[:-2], color=cluster_colors[cluster])
	ax_agg.plot(np.arange(5,7), cluster_mean[-2:], 'o', color=cluster_colors[cluster],)# label=cluster)
	ax_agg.errorbar(np.arange(5,7), cluster_mean[-2:], yerr=cluster_sem[-2:], color=cluster_colors[cluster])
	
	ax.legend(['correct reject',] + list(np.arange(5)+1), frameon=False, loc='upper left', bbox_to_anchor=(0, 1.05))
	ax.set_xlim(-400, 750)
	ax.set_xlabel('Time from image onset (ms)')
	ax.set_ylabel('Firing rate (Hz)')
	vbn_utils.formatFigure(fig, ax)


ax_agg.set_xticks(np.arange(7))
ax_agg.set_xticklabels(list(np.arange(5)+1) + ['hit', 'cr'])
ax_agg.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 0.2))
ax_agg.set_xlabel('Response time quintile')
ax_agg.set_ylabel('Baseline firing rate (Hz)')
vbn_utils.formatFigure(fig_agg, ax_agg)